In [ ]:
import csv
import os
import cobra
import numpy as np
import pandas as pd
from scipy.integrate import solve_ivp as solver
from cobra import Model, Reaction, Metabolite
from tqdm import tqdm
import matplotlib.pyplot as plt
from cobra.flux_analysis import pfba
import networkx as nx
import matplotlib.lines as mlines
from matplotlib.patches import FancyArrowPatch
import seaborn as sns
import joblib
from sklearn.metrics import confusion_matrix, accuracy_score, recall_score
from matplotlib.colors import ListedColormap
import matplotlib.patches as mpatches
import sys

YCFA_file = '/home/sli/personal/sli/butyrate/nutrints_YCFA.csv'
df = pd.read_csv(YCFA_file,header = 0)
YCFA_subs = df.iloc[:,0]
YCFA_con = df.iloc[:,2]

sub_list = []
for i in range(len(YCFA_subs)):
    sub_list.append(YCFA_subs[i])
    
Initial_state = []
for i in range(len(YCFA_con)):
    Initial_state.append(YCFA_con[i])
    
Initial_state = np.array(Initial_state)

def get_global_exchange_pool(models):

    global_pool = set()
    for m in models:
        global_pool.update([r.id for r in m.exchanges])
    return sorted(list(global_pool))


def mu_general_v2(model, concentration_vec, media_sub_ids, km, vmax, carbon):
    concentration_vec = np.nan_to_num(concentration_vec, nan=0.0, posinf=1000.0)
    concentration_vec = np.maximum(concentration_vec, 0)
    
    with model as m:
        for i, rxn_id in enumerate(media_sub_ids):
            if rxn_id not in m.exchanges:
                continue
            c = float(concentration_vec[i])
            if rxn_id == carbon:
                denom = c + km
                uptake_limit = (c / denom) * vmax if denom > 1e-9 else 0.0
            else:
                uptake_limit = c 
            m.reactions.get_by_id(rxn_id).lower_bound = -float(uptake_limit)

        try:
            sol = m.optimize()
            if sol.status != "optimal":
                zero_fluxes = pd.Series(0, index=[r.id for r in m.reactions])
                return [0.0] * len(media_sub_ids), zero_fluxes, 0.0
            
            ex_fluxes = [sol.fluxes[rid] if rid in m.exchanges else 0.0 for rid in media_sub_ids]
            full_fluxes = sol.fluxes 
            mu = max(sol.objective_value, 0.0)
            
            return ex_fluxes, full_fluxes, mu
        except:
            zero_fluxes = pd.Series(0, index=[r.id for r in m.reactions])
            return [0.0] * len(media_sub_ids), zero_fluxes, 0.0

def create_co_culture_medium_v2(models, carbon_id, carbon_con, 
                                YCFA_subs, sub_list, Initial_state, 
                                species_names, 
                                initial_biomass_list=None):

    all_possible_metabolites = get_global_exchange_pool(models)
    
    env_concs = {rid: 0.0 for rid in all_possible_metabolites}

    ycfa_lookup = dict(zip(sub_list, Initial_state))
    for rid in all_possible_metabolites:
        if rid in ycfa_lookup:
            env_concs[rid] = ycfa_lookup[rid]

    default_carbons = ['EX_glc_D(e)', 'EX_malt(e)', 'EX_cellb(e)']
    for c in default_carbons:
        if c in env_concs: env_concs[c] = 0.0
    
    env_concs[carbon_id] = carbon_con

    media_sub_ids = list(env_concs.keys())
    media_concs_vals = np.array([env_concs[rid] for rid in media_sub_ids])
    
    num_models = len(models)
    if initial_biomass_list is None:
        initial_biomass_list = [0.1] * num_models
    
    y0 = np.append(media_concs_vals, initial_biomass_list)
    biomass_names = [f"{name}_biomass" for name in species_names]
    
    df_columns = media_sub_ids + biomass_names
    all_con = pd.DataFrame([y0], columns=df_columns)
    all_con['t'] = 0
    
    return media_sub_ids, y0, all_con

def patch_general(dt, t_end, all_con, models, species_names, CO_YCFA_sub, km_list, vmax_list, carbon):
    t = 0
    biomass_cols = [f"{name}_biomass" for name in species_names]
    metabolite_cols = [c for c in all_con.columns if c not in biomass_cols + ['t']]

    all_con_list = [all_con]
    exchange_flux_list_dict = {name: [] for name in species_names}
    full_flux_list_dict = {name: [] for name in species_names}

    while t < t_end:
        t += dt
        current_row = all_con_list[-1].iloc[-1]
        concentration_vec = np.nan_to_num(current_row[metabolite_cols].values, nan=0.0)
        
        Mu_ex_list = []
        Mu_full_list = []
        Mu_val_list = []

        for i, model in enumerate(models):
            ex, full, mu = mu_general_v2(model, concentration_vec, metabolite_cols, km_list[i], vmax_list[i], carbon)
            Mu_ex_list.append(ex)
            Mu_full_list.append(full)
            Mu_val_list.append(mu)

        new_row_dict = {}
        for j, col in enumerate(metabolite_cols):
            delta = sum(dt * Mu_ex_list[i][j] * np.nan_to_num(current_row[f"{species_names[i]}_biomass"], nan=0.0) 
                        for i in range(len(species_names)))
            new_row_dict[col] = max(0.0, np.nan_to_num(current_row[col] + delta, nan=0.0))

        for i, name in enumerate(species_names):
            b_col = f"{name}_biomass"
            new_biomass = current_row[b_col] * (1 + Mu_val_list[i] * dt)
            new_row_dict[b_col] = max(0.0, np.nan_to_num(new_biomass, nan=0.0))

        new_row_dict['t'] = t
        all_con_list.append(pd.DataFrame([new_row_dict]))

        for i, name in enumerate(species_names):
            ex_row = {'t': t}
            for j, col in enumerate(metabolite_cols):
                ex_row[col] = Mu_ex_list[i][j]
            ex_row[f"{name}_biomass"] = Mu_val_list[i]
            exchange_flux_list_dict[name].append(pd.DataFrame([ex_row]))

            full_row = Mu_full_list[i].to_frame().T
            full_row['t'] = t
            full_flux_list_dict[name].append(full_row)

        if len(all_con_list) % 100 == 0:
            all_con_list = [pd.concat(all_con_list, ignore_index=True).copy()]

    final_all_con = pd.concat(all_con_list, ignore_index=True)
    
    final_exchange_flux_dict = {
        name: pd.concat(exchange_flux_list_dict[name], ignore_index=True) 
        for name in species_names
    }
    
    final_full_flux_dict = {
        name: pd.concat(full_flux_list_dict[name], ignore_index=True) 
        for name in species_names
    }

    return final_all_con, final_exchange_flux_dict, final_full_flux_dict

In [ ]:
Aca_old = cobra.io.read_sbml_model('/home/sli/personal/sli/butyrate/butyrate_models/Anaerostipes_caccae_DSM_14662.xml')
Bin_new = cobra.io.read_sbml_model('/home/sli/personal/sli/butyrate/butyrate_models/Bifidobacterium_longum_infantis_ATCC_15697.xml')
Blu_old = cobra.io.read_sbml_model('/home/sli/personal/sli/butyrate/butyrate_models/Blautia_luti_DSM_14534.xml')
Bbi_old = cobra.io.read_sbml_model('/home/sli/personal/sli/butyrate/butyrate_models/Bifidobacterium_bifidum_ATCC_29521.xml')
Cae_old = cobra.io.read_sbml_model("/home/sli/personal/sli/butyrate/butyrate_models/Collinsella_aerofaciens_ATCC_25986.xml")
Bfr_new = cobra.io.read_sbml_model('/home/sli/personal/sli/butyrate/butyrate_models/Bactorides_fragilis_638R_new.xml')
Bps_new = cobra.io.read_sbml_model('/home/sli/personal/sli/butyrate/butyrate_models/Bifidobacterium_pseudocatenulatum_DSM_20438.xml')
Bbr_new = cobra.io.read_sbml_model('/home/sli/personal/sli/butyrate/butyrate_models/Bifidobacterium_breve_DSM_20213.xml')

AC_model = Aca_old.copy()
BI_model = Bin_new.copy()
BL_model = Blu_old.copy()
Bb_model = Bbi_old.copy()
CA_model = Cae_old.copy()
BF_model = Bfr_new.copy()
BP_model = Bps_new.copy()
BB_model = Bbr_new.copy()

# Set lower bounds of each model to 0 to create in-silico YCFA growth environment for each model
for ex in AC_model.exchanges:
    ex.lower_bound = 0

for ex in BI_model.exchanges:
    ex.lower_bound = 0
    
for ex in BL_model.exchanges:
    ex.lower_bound = 0

for ex in Bb_model.exchanges:
    ex.lower_bound = 0

for ex in CA_model.exchanges:
    ex.lower_bound = 0

for ex in BF_model.exchanges:
    ex.lower_bound = 0

for ex in BP_model.exchanges:
    ex.lower_bound = 0

for ex in BB_model.exchanges:
    ex.lower_bound = 0

In [ ]:
%%time
models = [AC_model,BI_model,BL_model,Bb_model,CA_model,BF_model,BP_model,BB_model]
species_names = ['AC','BI','BL',"Bb",'CA','BF','BP','BB']
carbon = 'EX_2fuclac(e)'
km_list = [0.0,18.410,0.0,0.0,0.0,0.691,0.0,0.0]
vmax_list = [0.0,14.10,0.0,0.0,0.0,2.59,0.0,0.0]
CO_YCFA_sub, CO_YCFA_con, initial_df = create_co_culture_medium_v2(
    models=models,
    carbon_id=carbon,
    carbon_con=28,
    YCFA_subs=YCFA_subs,
    sub_list=sub_list,
    Initial_state=Initial_state,
    species_names=species_names,
    initial_biomass_list=[0.001]*len(models))
all_con_2FL, ex_flux_dict_2FL, full_flux_dict_2FL = patch_general(
    dt = 0.25,t_end = 72,
    all_con=initial_df,
    models=models,
    species_names=species_names,
    CO_YCFA_sub=CO_YCFA_sub,
    km_list=km_list,
    vmax_list=vmax_list,
    carbon = carbon
)

In [ ]:
%%time
models = [AC_model,BI_model,BL_model,Bb_model,CA_model,BF_model,BP_model,BB_model]
species_names = ['AC','BI','BL',"Bb",'CA','BF','BP','BB']
carbon = 'EX_3fuclac(e)'
km_list = [0.0,23.620,0.0,0.0,0.0,4.623,0.001,0.0]
vmax_list = [0.0,0.27,0.0,0.0,0.0,0.12,1.04,0.0]
CO_YCFA_sub, CO_YCFA_con, initial_df = create_co_culture_medium_v2(
    models=models,
    carbon_id=carbon,
    carbon_con=28,
    YCFA_subs=YCFA_subs,
    sub_list=sub_list,
    Initial_state=Initial_state,
    species_names=species_names,
    initial_biomass_list=[0.001]*len(models))
all_con_3FL, ex_flux_dict_3FL, full_flux_dict_3FL = patch_general(
    dt = 0.25,t_end = 72,
    all_con=initial_df,
    models=models,
    species_names=species_names,
    CO_YCFA_sub=CO_YCFA_sub,
    km_list=km_list,
    vmax_list=vmax_list,
    carbon = carbon
)

In [ ]:
%%time
models = [AC_model,BI_model,BL_model,Bb_model,CA_model,BF_model,BP_model,BB_model]
species_names = ['AC','BI','BL',"Bb",'CA','BF','BP','BB']
carbon = 'EX_dfuclac(e)'
km_list = [0.0,12.971,0.0,0.0,0.0,9.324,0.0,0.0]
vmax_list = [0.0,0.11,0.0,0.0,0.0,5.74,0.0,0.0]
CO_YCFA_sub, CO_YCFA_con, initial_df = create_co_culture_medium_v2(
    models=models,
    carbon_id=carbon,
    carbon_con=21.5,
    YCFA_subs=YCFA_subs,
    sub_list=sub_list,
    Initial_state=Initial_state,
    species_names=species_names,
    initial_biomass_list=[0.001]*len(models))
all_con_DFL, ex_flux_dict_DFL, full_flux_dict_DFL = patch_general(
    dt = 0.25,t_end = 72,
    all_con=initial_df,
    models=models,
    species_names=species_names,
    CO_YCFA_sub=CO_YCFA_sub,
    km_list=km_list,
    vmax_list=vmax_list,
    carbon = carbon
)

In [ ]:
%%time
models = [AC_model,BI_model,BL_model,Bb_model,CA_model,BF_model,BP_model,BB_model]
species_names = ['AC','BI','BL',"Bb",'CA','BF','BP','BB']
carbon = 'EX_3slac(e)'
km_list = [0.0,0.0,0.0,0.0,0.0,11.311,0.0,0.0]
vmax_list = [0.0,0.0,0.0,0.0,0.0,6.86,0.0,0.0]
CO_YCFA_sub, CO_YCFA_con, initial_df = create_co_culture_medium_v2(
    models=models,
    carbon_id=carbon,
    carbon_con=21.6,
    YCFA_subs=YCFA_subs,
    sub_list=sub_list,
    Initial_state=Initial_state,
    species_names=species_names,
    initial_biomass_list=[0.001]*len(models))
all_con_3SL, ex_flux_dict_3SL, full_flux_dict_3SL = patch_general(
    dt = 0.25,t_end = 72,
    all_con=initial_df,
    models=models,
    species_names=species_names,
    CO_YCFA_sub=CO_YCFA_sub,
    km_list=km_list,
    vmax_list=vmax_list,
    carbon = carbon
)

In [ ]:
%%time
models = [AC_model,BI_model,BL_model,Bb_model,CA_model,BF_model,BP_model,BB_model]
species_names = ['AC','BI','BL',"Bb",'CA','BF','BP','BB']
carbon = 'EX_6slac(e)'
km_list = [0.0,0.0,0.0,0.0,0.0,19.311,0.0,0.0]
vmax_list = [0.0,0.0,0.0,0.0,0.0,10.12,0.0,0.0]
CO_YCFA_sub, CO_YCFA_con, initial_df = create_co_culture_medium_v2(
    models=models,
    carbon_id=carbon,
    carbon_con=21.6,
    YCFA_subs=YCFA_subs,
    sub_list=sub_list,
    Initial_state=Initial_state,
    species_names=species_names,
    initial_biomass_list=[0.001]*len(models))
all_con_6SL, ex_flux_dict_6SL, full_flux_dict_6SL = patch_general(
    dt = 0.25,t_end = 72,
    all_con=initial_df,
    models=models,
    species_names=species_names,
    CO_YCFA_sub=CO_YCFA_sub,
    km_list=km_list,
    vmax_list=vmax_list,
    carbon = carbon
)

In [ ]:
%%time
models = [AC_model,BI_model,BL_model,Bb_model,CA_model,BF_model,BP_model,BB_model]
species_names = ['AC','BI','BL',"Bb",'CA','BF','BP','BB']
carbon = 'EX_lacnttr(e)'
km_list = [0.0,15.382,0.0,0.0,0.0,9.311,0.0,27.300]
vmax_list = [0.0,7.83,0.0,0.0,0.0,5.82,0.0,0.97]
CO_YCFA_sub, CO_YCFA_con, initial_df = create_co_culture_medium_v2(
    models=models,
    carbon_id=carbon,
    carbon_con=20,
    YCFA_subs=YCFA_subs,
    sub_list=sub_list,
    Initial_state=Initial_state,
    species_names=species_names,
    initial_biomass_list=[0.001]*len(models))
all_con_LNT, ex_flux_dict_LNT, full_flux_dict_LNT = patch_general(
    dt = 0.25,t_end = 72,
    all_con=initial_df,
    models=models,
    species_names=species_names,
    CO_YCFA_sub=CO_YCFA_sub,
    km_list=km_list,
    vmax_list=vmax_list,
    carbon = carbon
)

In [ ]:
%%time
models = [AC_model,BI_model,BL_model,Bb_model,CA_model,BF_model,BP_model,BB_model]
species_names = ['AC','BI','BL',"Bb",'CA','BF','BP','BB']
carbon = 'EX_lacnnttr(e)'
km_list = [0.0,19.770,0.0,0.0,0.0,9.850,1.389,27.300]
vmax_list = [0.0,16.07,0.0,0.0,0.0,10.41,3.62,0.97]
CO_YCFA_sub, CO_YCFA_con, initial_df = create_co_culture_medium_v2(
    models=models,
    carbon_id=carbon,
    carbon_con=20,
    YCFA_subs=YCFA_subs,
    sub_list=sub_list,
    Initial_state=Initial_state,
    species_names=species_names,
    initial_biomass_list=[0.001]*len(models))
all_con_LNNT, ex_flux_dict_LNNT, full_flux_dict_LNNT = patch_general(
    dt = 0.25,t_end = 72,
    all_con=initial_df,
    models=models,
    species_names=species_names,
    CO_YCFA_sub=CO_YCFA_sub,
    km_list=km_list,
    vmax_list=vmax_list,
    carbon = carbon
)

In [ ]:
save_path = '8species_results'
if not os.path.exists(save_path): os.makedirs(save_path)

all_species_2FL = {
    'all_con': all_con_2FL,
    'ex_flux': ex_flux_dict_2FL,
    'full_flux': full_flux_dict_2FL
}
joblib.dump(all_species_2FL,os.path.join(save_path,'/home/sli/personal/sli/butyrate/8speices/all_species_2FL_results.gz'),compress=3)

all_species_3FL = {
    'all_con': all_con_3FL,
    'ex_flux': ex_flux_dict_3FL,
    'full_flux': full_flux_dict_3FL
}
joblib.dump(all_species_3FL,os.path.join(save_path,'/home/sli/personal/sli/butyrate/8speices/all_species_3FL_results.gz'),compress=3)

all_species_3SL = {
    'all_con': all_con_3SL,
    'ex_flux': ex_flux_dict_3SL,
    'full_flux': full_flux_dict_3SL
}
joblib.dump(all_species_3SL,os.path.join(save_path,'/home/sli/personal/sli/butyrate/8speices/all_species_3SL_results.gz'),compress=3)

all_species_6SL = {
    'all_con': all_con_6SL,
    'ex_flux': ex_flux_dict_6SL,
    'full_flux': full_flux_dict_6SL
}
joblib.dump(all_species_6SL,os.path.join(save_path,'/home/sli/personal/sli/butyrate/8speices/all_species_6SL_results.gz'),compress=3)

all_species_DFL = {
    'all_con': all_con_DFL,
    'ex_flux': ex_flux_dict_DFL,
    'full_flux': full_flux_dict_DFL
}
joblib.dump(all_species_DFL,os.path.join(save_path,'/home/sli/personal/sli/butyrate/8speices/all_species_DFL_results.gz'),compress=3)

all_species_LNT = {
    'all_con': all_con_LNT,
    'ex_flux': ex_flux_dict_LNT,
    'full_flux': full_flux_dict_LNT
}
joblib.dump(all_species_LNT,os.path.join(save_path,'/home/sli/personal/sli/butyrate/8speices/all_species_LNT_results.gz'),compress=3)

all_species_LNNT = {
    'all_con': all_con_LNNT,
    'ex_flux': ex_flux_dict_LNNT,
    'full_flux': full_flux_dict_LNNT
}
joblib.dump(all_species_LNNT,os.path.join(save_path,'/home/sli/personal/sli/butyrate/8speices/all_species_LNNT_results.gz'),compress=3)

In [ ]:
file_path = '~/Documents/simulations/co-species cultivation/8species_results/all_species_2FL_results.gz'
file_path = os.path.expanduser(file_path)
all_species_2FL = joblib.load(file_path)
all_con_2FL = all_species_2FL['all_con']
ex_flux_2FL = all_species_2FL['ex_flux']
full_flux_2Fl = all_species_2FL['full_flux']

file_path = '~/Documents/simulations/co-species cultivation/8species_results/all_species_3FL_results.gz'
file_path = os.path.expanduser(file_path)
all_species_3FL = joblib.load(file_path)
all_con_3FL = all_species_3FL['all_con']
ex_flux_3FL = all_species_3FL['ex_flux']
full_flux_3Fl = all_species_3FL['full_flux']

file_path = '~/Documents/simulations/co-species cultivation/8species_results/all_species_3SL_results.gz'
file_path = os.path.expanduser(file_path)
all_species_3SL = joblib.load(file_path)
all_con_3SL = all_species_3SL['all_con']
ex_flux_3SL = all_species_3SL['ex_flux']
full_flux_3Sl = all_species_3SL['full_flux']

file_path = '~/Documents/simulations/co-species cultivation/8species_results/all_species_6SL_results.gz'
file_path = os.path.expanduser(file_path)
all_species_6SL = joblib.load(file_path)
all_con_6SL = all_species_6SL['all_con']
ex_flux_6SL = all_species_6SL['ex_flux']
full_flux_6Sl = all_species_6SL['full_flux']

file_path = '~/Documents/simulations/co-species cultivation/8species_results/all_species_DFL_results.gz'
file_path = os.path.expanduser(file_path)
all_species_DFL = joblib.load(file_path)
all_con_DFL = all_species_DFL['all_con']
ex_flux_DFL = all_species_DFL['ex_flux']
full_flux_DFl = all_species_DFL['full_flux']

file_path = '~/Documents/simulations/co-species cultivation/8species_results/all_species_LNT_results.gz'
file_path = os.path.expanduser(file_path)
all_species_LNT = joblib.load(file_path)
all_con_LNT = all_species_LNT['all_con']
ex_flux_LNT = all_species_LNT['ex_flux']
full_flux_LNT = all_species_LNT['full_flux']

file_path = '~/Documents/simulations/co-species cultivation/8species_results/all_species_LNNT_results.gz'
file_path = os.path.expanduser(file_path)
all_species_LNNT = joblib.load(file_path)
all_con_LNNT = all_species_LNNT['all_con']
ex_flux_LNNT = all_species_LNNT['ex_flux']
full_flux_LNNT = all_species_LNNT['full_flux']

In [ ]:
results_dir = os.path.expanduser('~/Documents/simulations/co-species cultivation/8species_results')

carbons = ['2FL', '3FL', 'DFL', '3SL', '6SL', 'LNT', 'LNNT']
species = ['AC_biomass', 'BI_biomass', 'BL_biomass', 'Bb_biomass', 'CA_biomass', 'BF_biomass', 'BP_biomass', 'BB_biomass']
scfas = ['EX_succ(e)', 'EX_lac_L(e)', 'EX_for(e)', 'EX_ac(e)', 'EX_ppa(e)', 'EX_but(e)']

biomass_dict = {}
scfa_dict = {}

for carbon in carbons:
    file_name = f'all_species_{carbon}_results.gz'
    file_path = os.path.join(results_dir, file_name)
    
    if os.path.exists(file_path):
        data = joblib.load(file_path)
        df_con = data['all_con']
        
        biomass_dict[carbon] = df_con[species].iloc[-1]
        
        scfa_dict[carbon] = df_con[scfas].iloc[-1] - df_con[scfas].iloc[0]
    else:
        print(f"Can't find file: {file_name}")

all_species_biomass_HMOs = pd.DataFrame(biomass_dict)
scfas_HMOs = pd.DataFrame(scfa_dict)

In [ ]:
all_species_biomass_HMOs_rel = all_species_biomass_HMOs / all_species_biomass_HMOs.sum()
all_species_biomass_HMOs_rel.index = ['A. caccae','B. infantis','B. luti','B. bifidum',
                                      'C. aerofaciens', 'B. fragilis', 'B. pseudocatenulatum',
                                      'B. breve']

scfas_HMOs.iloc[0,0] = 0
scfas_HMOs.iloc[0,1] = 0
scfas_HMOs.iloc[-1,1] = 0
scfas_HMOs.iloc[0,-1] = 0
scfas_HMOs.iloc[1,-1] = 0
scfas_HMOs.index = ['Succinate','Lactate','Formate','Acetate','Propionate','Butyrate']

In [ ]:
species = ['AC_biomass','BI_biomass','BL_biomass','Bb_biomass','CA_biomass','BF_biomass','BP_biomass','BB_biomass']
Biomass_2FL = all_con_2FL[species].iloc[-1]
Biomass_3FL = all_con_3FL[species].iloc[-1]
Biomass_3SL = all_con_3SL[species].iloc[-1]
Biomass_6SL = all_con_6SL[species].iloc[-1]
Biomass_DFL = all_con_DFL[species].iloc[-1]
Biomass_LNT = all_con_LNT[species].iloc[-1]
Biomass_LNNT = all_con_LNNT[species].iloc[-1]

In [ ]:
all_species_biomass_HMOs = pd.DataFrame({
    "2FL": Biomass_2FL,
    "3FL": Biomass_3FL,
    "3SL": Biomass_3SL,
    "6SL": Biomass_6SL,
    "DFL": Biomass_DFL,
    "LNT": Biomass_LNT,
    "LNNT": Biomass_LNNT
})

In [ ]:
all_species_biomass_HMOs_rel = all_species_biomass_HMOs / all_species_biomass_HMOs.sum()

In [ ]:
all_species_biomass_HMOs_rel.index = ['A. caccae','B. infantis','B. luti','B. bifidum',
                                      'C. aerofaciens', 'B. fragilis', 'B. pseudocatenulatum',
                                      'B. breve']

In [ ]:
scfas = ['EX_succ(e)','EX_lac_L(e)','EX_for(e)','EX_ac(e)','EX_ppa(e)','EX_but(e)']

In [ ]:
scfas_2FL = all_con_2FL[scfas].iloc[-1] - all_con_2FL[scfas].iloc[0]
scfas_3FL = all_con_3FL[scfas].iloc[-1] - all_con_3FL[scfas].iloc[0]
scfas_3SL = all_con_3SL[scfas].iloc[-1] - all_con_3SL[scfas].iloc[0]
scfas_6SL = all_con_6SL[scfas].iloc[-1] - all_con_6SL[scfas].iloc[0]
scfas_DFL = all_con_DFL[scfas].iloc[-1] - all_con_DFL[scfas].iloc[0]
scfas_LNT = all_con_LNT[scfas].iloc[-1] - all_con_LNT[scfas].iloc[0]
scfas_LNNT = all_con_LNNT[scfas].iloc[-1] - all_con_LNNT[scfas].iloc[0]

In [ ]:
scfas_HMOs = pd.DataFrame({
    "2FL": scfas_2FL,
    "3FL": scfas_3FL,
    "3SL": scfas_3SL,
    "6SL": scfas_6SL,
    "DFL": scfas_DFL,
    "LNT": scfas_LNT,
    "LNNT": scfas_LNNT
})

In [ ]:
scfas_HMOs.index = ['Succinate','Lactate','Formate','Acetate','Propionate','Butyrate']

In [ ]:
file_path_16S = '/Users/lishijia/Documents/simulations/Butyrate project/8species_16S.csv'
df_16S = pd.read_csv(file_path_16S,index_col=0)

file_path_scfa = '/Users/lishijia/Documents/simulations/Butyrate project/8species_scfa.csv'
df_scfa = pd.read_csv(file_path_scfa,index_col=0)

In [ ]:
common_species = df_16S.index.intersection(all_species_biomass_HMOs_rel.index)
common_hmos = df_16S.columns.intersection(all_species_biomass_HMOs_rel.columns)

In [ ]:
df_16s_aligned = df_16S.loc[common_species, common_hmos]
df_sim_aligned = all_species_biomass_HMOs_rel.loc[common_species, common_hmos]

df1 = df_16s_aligned.T
df2 = df_sim_aligned.T

In [ ]:
common_species_bar = df_16S.index.intersection(all_species_biomass_HMOs_rel.index)
common_hmos_bar = df_16S.columns.intersection(all_species_biomass_HMOs_rel.columns)

df1_aligned = df_16S.loc[common_species_bar, common_hmos_bar].T 
df2_aligned = all_species_biomass_HMOs_rel.loc[common_species_bar, common_hmos_bar].T 

cmap_set3 = plt.get_cmap('Set3')
colors_species = [cmap_set3(i) for i in np.linspace(0, 1, len(common_species_bar))]
color_dict_species = dict(zip(common_species_bar, colors_species))
plot_colors_species = [color_dict_species[species] for species in df1_aligned.columns] 

common_hmos_scfa = df_scfa.index.intersection(scfas_HMOs.index)
common_scfas_scfa = df_scfa.columns.intersection(scfas_HMOs.columns)

df_exp_scfa = df_scfa.loc[common_hmos_scfa, common_scfas_scfa]
df_sim_scfa = scfas_HMOs.loc[common_hmos_scfa, common_scfas_scfa]

exp_threshold_scfa = 0.0   
sim_threshold_scfa = 1e-4  

exp_bin_scfa = (df_exp_scfa > exp_threshold_scfa).astype(int)
sim_bin_scfa = (df_sim_scfa > sim_threshold_scfa).astype(int)

y_true_scfa = exp_bin_scfa.values.flatten()
y_pred_scfa = sim_bin_scfa.values.flatten()

TP_scfa = np.sum((y_true_scfa == 1) & (y_pred_scfa == 1))
TN_scfa = np.sum((y_true_scfa == 0) & (y_pred_scfa == 0))
FP_scfa = np.sum((y_true_scfa == 0) & (y_pred_scfa == 1))
FN_scfa = np.sum((y_true_scfa == 1) & (y_pred_scfa == 0))

accuracy_scfa = (TP_scfa + TN_scfa) / len(y_true_scfa) * 100
sensitivity_scfa = (TP_scfa / (TP_scfa + FN_scfa) * 100) if (TP_scfa + FN_scfa) > 0 else 0
title_str_scfa = f"Acc: {accuracy_scfa:.1f}% | Sens: {sensitivity_scfa:.1f}%" 

compare_matrix_scfa = exp_bin_scfa * 2 + sim_bin_scfa
plot_data_scfa = compare_matrix_scfa

colors_cm = ['#f0f0f0', '#e41a1c', '#ff7f00', '#2171b5']
cmap_cm = ListedColormap(colors_cm)

fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(22, 8), sharey=False)

df1_aligned.plot(kind='bar', stacked=True, ax=axes[0], color=plot_colors_species, 
                 edgecolor='black', linewidth=0.5, width=0.7)

axes[0].set_title('Measured Strain composition', fontsize=20, fontweight='bold', pad=15)
axes[0].set_ylabel('Relative Abundance', fontsize=18, fontweight='bold')
axes[0].set_ylim(0, 1) 
axes[0].tick_params(axis='x', rotation=45, labelsize=16)
axes[0].tick_params(axis='y', labelsize=16)
axes[0].legend_.remove() 

df2_aligned.plot(kind='bar', stacked=True, ax=axes[1], color=plot_colors_species, 
                 edgecolor='black', linewidth=0.5, width=0.7)

axes[1].set_title('Simulated Biomass', fontsize=20, fontweight='bold', pad=15)
axes[1].set_ylabel('Relative Biomass', fontsize=18, fontweight='bold')
axes[1].tick_params(axis='x', rotation=45, labelsize=16)
axes[1].tick_params(axis='y', labelsize=16)
axes[1].set_ylim(0, 1) 
axes[1].legend_.remove() 

sns.heatmap(plot_data_scfa, cmap=cmap_cm, cbar=False, ax=axes[2],
            linewidths=1.5, linecolor='white', 
            vmin=-0.5, vmax=3.5)

axes[2].set_title(title_str_scfa, fontsize=20, fontweight='bold', pad=15)
axes[2].tick_params(axis='x', rotation=45, labelsize=16)
axes[2].tick_params(axis='y', rotation=0, labelsize=16)

labels = ['A', 'B', 'C']
for i, ax in enumerate(axes):
    ax.text(-0.12, 1.05, labels[i], transform=ax.transAxes, 
            fontsize=26, fontweight='bold', va='bottom', ha='right')

species_handles = [mpatches.Patch(color=color_dict_species[sp], label=sp) for sp in df1_aligned.columns]

leg1 = axes[2].legend(handles=species_handles, title='Species', 
                      bbox_to_anchor=(1.05, 1.0), loc='upper left', 
                      ncol=1, fontsize=15, title_fontsize=17, frameon=False)
axes[2].add_artist(leg1)

tn_patch = mpatches.Patch(color='#f0f0f0', label='True Negative (Both Non-producer)')
tp_patch = mpatches.Patch(color='#2171b5', label='True Positive (Both Producer)')
fp_patch = mpatches.Patch(color='#e41a1c', label='False Positive (Sim Predicted, Exp Not)')
fn_patch = mpatches.Patch(color='#ff7f00', label='False Negative (Exp Produced, Sim Not)')

handles_scfa = [tp_patch, tn_patch, fp_patch, fn_patch]

leg2 = axes[2].legend(handles=handles_scfa, title='Prediction Status', 
                      bbox_to_anchor=(1.05, 0.42), loc='upper left', 
                      ncol=1, fontsize=14, title_fontsize=16, frameon=False)

plt.tight_layout(w_pad=3.0)

plt.show()
fig.savefig("/Users/lishijia/Documents/simulations/Butyrate project/Figure_5.jpeg", format="jpeg", dpi=900, bbox_inches="tight")

In [ ]:
import os
import joblib
from pathlib import Path

root_path = Path("/Users/lishijia/Documents/simulations/co-species cultivation/")

all_sim_data = {}

for folder in root_path.glob("*_results"):
    if folder.is_dir():
        print(f"Scanning: {folder.name}")
        
        for file_path in folder.glob("*_results.gz"):
            prefix = file_path.name.replace("_results.gz", "")
            
            try:
                data = joblib.load(file_path)
                all_sim_data[prefix] = data
            except Exception as e:
                print(f"Failed at loading {file_path.name}: {e}")

In [ ]:
import string
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']
plt.rcParams['svg.fonttype'] = 'none'

carbon_sources = ['LNT', 'LNNT', '2FL', '3FL', 'DFL']
dt = 0.25

five_species_data = {}
for cs in carbon_sources:
    target_cs = cs if cs != 'LNNT' else 'LNnT'
    
    key_1 = f"Aca_Bin_3nohmo_{cs}"
    key_2 = f"Aca_Bin_3nohmo_{target_cs}"
    
    if key_1 in all_sim_data:
        five_species_data[cs] = (all_sim_data[key_1].get('all_con', pd.DataFrame()), all_sim_data[key_1].get('ex_flux', {}))
    elif key_2 in all_sim_data:
        five_species_data[cs] = (all_sim_data[key_2].get('all_con', pd.DataFrame()), all_sim_data[key_2].get('ex_flux', {}))
    else:
        five_species_data[cs] = (pd.DataFrame(), {})

invaders = [
    ('BL', 'Blu'), 
    ('Bb', 'Bbi'), 
    ('CA', 'Cae')
]

panel_labels = string.ascii_uppercase 
label_idx = 0 

fig = plt.figure(figsize=(28, 36), dpi=300)

gs = GridSpec(5, 4, figure=fig, hspace=0.95, wspace=0.35)

for row_idx, cs in enumerate(carbon_sources):
    df_5sp, flux_5sp = five_species_data[cs]

    ax_but = fig.add_subplot(gs[row_idx, 0])
    
    if df_5sp.empty:
        ax_but.text(0.5, 0.5, "Data Missing", ha='center', color='red', fontsize=21)
        ax_but.set_xticks([]); ax_but.set_yticks([])
        label_idx += 1
    else:
        t_5_raw = df_5sp['Time'] if 'Time' in df_5sp.columns else df_5sp.index
        t_5 = np.array(t_5_raw, dtype=float) * dt
        
        but_5 = np.maximum(0, df_5sp.get('EX_but(e)', np.zeros(len(t_5))))
        min_len_but = min(len(t_5), len(but_5))
        
        baseline_con_found = False
        if 'all_sim_data' in globals():
            possible_keys = [f'Aca_Bin_{cs}', f'Aca_Bin_{cs.replace("LNNT", "LNnT")}']
            for p_key in possible_keys:
                if p_key in all_sim_data:
                    df_2sp = all_sim_data[p_key]['all_con']
                    t_2_raw = df_2sp['Time'] if 'Time' in df_2sp.columns else df_2sp.index
                    t_2 = np.array(t_2_raw, dtype=float) * dt
                    
                    but_2 = np.maximum(0, df_2sp.get('EX_but(e)', np.zeros(len(t_2))))
                    min_len_but2 = min(len(t_2), len(but_2))
                    
                    ax_but.plot(t_2[:min_len_but2], but_2[:min_len_but2], label='Baseline ($Aca + Bin$)', 
                                color='#d62728', linestyle='--', linewidth=3.5, alpha=0.8)
                    baseline_con_found = True
                    break
                    
        if not baseline_con_found:
            print(f"Didn't find the dual-strain baseline concentartion data in all_sim_data: {cs}")

        ax_but.plot(t_5[:min_len_but], but_5[:min_len_but], label='5-Species Combination', color='#d62728', linewidth=3.5)

        ax_but.text(-0.06, 1.08, panel_labels[label_idx], transform=ax_but.transAxes, fontsize=28, fontweight='bold', va='bottom', ha='right')
        ax_but.set_title(f'Butyrate on {cs}', loc='center', fontsize=27, fontweight='bold')
        ax_but.set_xlabel('Time (h)', fontsize=23, fontweight='bold')

        ax_but.set_ylabel('Butyrate (mM)', fontsize=24, fontweight='bold')
            
        ax_but.legend(loc='upper center', bbox_to_anchor=(0.5, -0.22), ncol=1, 
                       fontsize=21, frameon=False, columnspacing=1.0)
        
        ax_but.grid(True, alpha=0.3)
        ax_but.tick_params(axis='both', labelsize=22)
        label_idx += 1

    for inv_idx, (sp_key, sp_name) in enumerate(invaders):
        col_idx = inv_idx + 1 # 列索引为 1, 2, 3
        ax_flux = fig.add_subplot(gs[row_idx, col_idx])
        
        actual_sp_key = sp_key if sp_key in flux_5sp else sp_key.upper()
        
        if not df_5sp.empty and actual_sp_key in flux_5sp:
            df_sp = flux_5sp[actual_sp_key]
            t_5_raw = df_5sp['Time'] if 'Time' in df_5sp.columns else df_sp.index
            t_5 = np.array(t_5_raw, dtype=float) * dt 

            ac_flux_base, lac_flux_base, t_2_base = [], [], []
            baseline_flux_found = False
            
            if 'all_sim_data' in globals():
                possible_keys = [f'Aca_Bin_{cs}', f'Aca_Bin_{cs.replace("LNNT", "LNnT")}']
                for p_key in possible_keys:
                    if p_key in all_sim_data and 'ex_flux' in all_sim_data[p_key] and 'AC' in all_sim_data[p_key]['ex_flux']:
                        df_ex_base = all_sim_data[p_key]['ex_flux']['AC']
                        t_2_raw_base = all_sim_data[p_key]['all_con'].get('Time', all_sim_data[p_key]['all_con'].index)
                        t_2_base = np.array(t_2_raw_base, dtype=float) * dt
                        
                        ac_flux_base = df_ex_base.get('EX_ac(e)', np.zeros(len(t_2_base)))
                        lac_name_base = 'EX_lac_L(e)' if 'EX_lac_L(e)' in df_ex_base.columns else 'EX_lac(e)'
                        lac_flux_base = df_ex_base.get(lac_name_base, np.zeros(len(t_2_base)))
                        
                        baseline_flux_found = True
                        break

            ac_flux = df_sp.get('EX_ac(e)', np.zeros(len(t_5)))
            lac_name = 'EX_lac_L(e)' if 'EX_lac_L(e)' in df_sp.columns else 'EX_lac(e)'
            lac_flux = df_sp.get(lac_name, np.zeros(len(t_5)))
            
            ac_flux_ac, lac_flux_ac = np.zeros(len(t_5)), np.zeros(len(t_5))
            if 'AC' in flux_5sp:
                df_ac = flux_5sp['AC']
                ac_flux_ac = df_ac.get('EX_ac(e)', np.zeros(len(t_5)))
                lac_name_ac = 'EX_lac_L(e)' if 'EX_lac_L(e)' in df_ac.columns else 'EX_lac(e)'
                lac_flux_ac = df_ac.get(lac_name_ac, np.zeros(len(t_5)))

            min_len_inv = min(len(t_5), len(ac_flux))
            ax_flux.plot(t_5[:min_len_inv], ac_flux[:min_len_inv], label=f'${sp_name}$ Acetate', color='#ff7f0e', linewidth=2.5)
            ax_flux.plot(t_5[:min_len_inv], lac_flux[:min_len_inv], label=f'${sp_name}$ Lactate', color='#2ca02c', linewidth=2.5)
            
            min_len_ac5 = min(len(t_5), len(ac_flux_ac))
            ax_flux.plot(t_5[:min_len_ac5], ac_flux_ac[:min_len_ac5], label='$Aca$ Acetate (5sp)', color='#ff7f0e', linewidth=2.5, linestyle='--', alpha=0.8)
            ax_flux.plot(t_5[:min_len_ac5], lac_flux_ac[:min_len_ac5], label='$Aca$ Lactate (5sp)', color='#2ca02c', linewidth=2.5, linestyle='--', alpha=0.8)
            
            if baseline_flux_found:
                min_len_base = min(len(t_2_base), len(ac_flux_base))
                ax_flux.plot(t_2_base[:min_len_base], ac_flux_base[:min_len_base], label='$Aca$ Acetate (Base)', 
                             color='#ff7f0e', linewidth=1.5, linestyle=':', marker='o', markersize=3, markevery=5, alpha=0.6)
                ax_flux.plot(t_2_base[:min_len_base], lac_flux_base[:min_len_base], label='$Aca$ Lactate (Base)', 
                             color='#2ca02c', linewidth=1.5, linestyle=':', marker='o', markersize=3, markevery=5, alpha=0.6)

            ax_flux.axhline(0, color='black', linewidth=1)

            y_pts_ac = [np.min(ac_flux), np.max(ac_flux), np.min(ac_flux_ac), np.max(ac_flux_ac)]
            y_pts_lac = [np.min(lac_flux), np.max(lac_flux), np.min(lac_flux_ac), np.max(lac_flux_ac)]
            if baseline_flux_found:
                y_pts_ac.extend([np.min(ac_flux_base), np.max(ac_flux_base)])
                y_pts_lac.extend([np.min(lac_flux_base), np.max(lac_flux_base)])

            y_min_data = min(min(y_pts_ac), min(y_pts_lac), 0)
            y_max_data = max(max(y_pts_ac), max(y_pts_lac), 0)
            
            margin = (y_max_data - y_min_data) * 0.15
            if margin == 0: margin = 1.0  
            y_lower = y_min_data - margin
            y_upper = y_max_data + margin
            
            if np.min(ac_flux) < -0.1:
                ax_flux.fill_between(t_5[:min_len_inv], y_upper, y_lower, where=(ac_flux[:min_len_inv] < -0.1), 
                                     color='#ff7f0e', alpha=0.15, label='Acetate Compete')
            if np.min(lac_flux) < -0.1:
                ax_flux.fill_between(t_5[:min_len_inv], y_upper, y_lower, where=(lac_flux[:min_len_inv] < -0.1), 
                                     color='#2ca02c', alpha=0.15, label='Lactate Compete')
                
            ax_flux.text(-0.06, 1.08, panel_labels[label_idx], transform=ax_flux.transAxes, fontsize=28, fontweight='bold', va='bottom', ha='right')
            ax_flux.set_title(f'${sp_name}$ fluxes on {cs}', loc='center', fontsize=27, fontweight='bold')
            ax_flux.set_xlabel('Time (h)', fontsize=23, fontweight='bold')

            if col_idx == 1:
                ax_flux.set_ylabel('Flux (mmol/gDW/h)', fontsize=24, fontweight='bold')
                
            ax_flux.legend(loc='upper center', bbox_to_anchor=(0.5, -0.22), ncol=2, 
                            fontsize=19, frameon=False, columnspacing=0.8)
                
            ax_flux.grid(True, alpha=0.3)
            ax_flux.tick_params(axis='both', labelsize=22)
            ax_flux.set_ylim(y_lower, y_upper) 
        else:
            ax_flux.text(0.5, 0.5, f"Missing {sp_key}", ha='center', color='red', fontsize=25)
            ax_flux.set_xticks([]); ax_flux.set_yticks([])
            
        label_idx += 1

plt.subplots_adjust(top=0.94, bottom=0.08, left=0.05, right=0.98)
fig.savefig("/Users/lishijia/Documents/simulations/Butyrate project/Combined_Figure5_Transposed.jpeg", format="jpeg", dpi=900, bbox_inches="tight")
plt.show()